# NB06 — Method Ablation on Potsdam val tile 6_15

Compares 4 methods in one pass. Backbone runs **once per crop**, shared across all methods.

| # | Method | Description |
|---|---|---|
| A | ZS-single | Zero-shot, one word per class (original SegEarth-OV-3 style) |
| B | ZS-multi | Zero-shot, multi-synonym prompts (our `cls_potsdam.txt`) |
| C | ZS-multi + PAMR | Method B + PAMR boundary refinement (image-guided edge sharpening) |
| D | PTSAM + PAMR | Multi-synonym + soft prompts + PAMR |

**Eval:** 5-class mIoU (impervious, building, low_veg, tree, car) — excludes clutter, matching literature (57.8%).

**Datasets:** sam3-weights · 6isprs · ptsam-soft-prompts

## 1 — Environment setup (~5 min)

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop('PYTHONPATH', None)
os.environ['PATH'] = '/tmp/miniconda/bin:' + os.environ['PATH']

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [ ]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 tifffile matplotlib -q

## 2 — Clone repo + install model dependencies

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path('/tmp/SegEarth-OV-3')

if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
    print(f'Updated -> {REPO}')
else:
    subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/HarishDeepak/rg-segearth-ov3', str(REPO)],
        check=True)
    print(f'Cloned -> {REPO}')

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Ablation: 4 methods on val tile 6_15

- Backbone runs **once per crop** (shared across A/B/C/D)
- PAMR iterates 10× over soft logits using RGB image as affinity guide
- All scores use `segearthov3_segmentor._inference_single_view` formula

In [ ]:
%%bash
export MPLBACKEND=Agg
export PYTHONUNBUFFERED=1
source /tmp/miniconda/bin/activate segearth
cd /tmp/SegEarth-OV-3

python - << 'PYEOF'
import sys, torch, torch.nn.functional as F
import numpy as np, tifffile
from pathlib import Path
from PIL import Image

sys.stdout.reconfigure(line_buffering=True)

DEVICE               = "cuda"
POTSDAM_ROOT         = Path("/kaggle/input/datasets/dummyirl/6isprs")
VAL_TILE             = "6_15"
N_CLASSES            = 6
BG_IDX               = 5
IGNORE_IDX           = 255
CROP_SIZE            = 1024
STRIDE               = 512
CONFIDENCE_THRESHOLD = 0.1
PROB_THD             = 0.1
PAMR_ITER            = 10    # PAMR boundary refinement iterations
PAMR_DILATIONS       = [1, 2, 4, 8, 12, 24]   # multi-scale local affinity

# ── PAMR (boundary refinement using RGB image affinity) ────────────────────
# pamr.py is in the repo root — refines soft logits using image edge guidance.
# Higher PAMR_ITER = sharper edges but slower. 10 is the standard.
import sys as _sys
_sys.path.insert(0, str(Path('.')))
from pamr import PAMR
pamr_module = PAMR(num_iter=PAMR_ITER, dilations=PAMR_DILATIONS).to(DEVICE).eval()
print(f"PAMR loaded (iter={PAMR_ITER}, dilations={PAMR_DILATIONS})", flush=True)

# ── load soft prompts ──────────────────────────────────────────────────────
candidates = list(Path("/kaggle/input").rglob("soft_prompts.pt"))
if not candidates:
    candidates = [p for p in Path("/kaggle/input").rglob("*.pt") if "sam3" not in p.name]
if not candidates:
    print("WARNING: soft_prompts.pt not found — methods C/D will skip PTSAM", flush=True)
    soft_prompts = None
else:
    soft_prompts = torch.load(str(candidates[0]), weights_only=True).to(DEVICE)
    print(f"soft_prompts: {soft_prompts.shape}", flush=True)

# ── load model ─────────────────────────────────────────────────────────────
from config_local import SAM3_CHECKPOINT
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("Loading SAM3...", flush=True)
model = build_sam3_image_model(
    bpe_path="./sam3/assets/bpe_simple_vocab_16e6.txt.gz",
    checkpoint_path=SAM3_CHECKPOINT, device=DEVICE)
model.eval()
for p in model.parameters(): p.requires_grad = False
print(f"GPU: {torch.cuda.get_device_name(0)}", flush=True)
processor = Sam3Processor(model, confidence_threshold=CONFIDENCE_THRESHOLD, device=DEVICE)

# ── prompt sets ────────────────────────────────────────────────────────────
# Method A: single canonical word per class
SINGLE_WORDS   = ["impervious surface", "building", "low vegetation", "tree", "car"]
SINGLE_CLS_IDX = list(range(5))

def get_multi_synonyms(path):
    words, indices = [], []
    for cls_idx, line in enumerate(Path(path).read_text().splitlines()):
        line = line.strip()
        if not line: continue
        for syn in line.split(','):
            syn = syn.strip()
            if syn:
                words.append(syn); indices.append(cls_idx)
    return words, indices

MULTI_WORDS, MULTI_CLS_IDX = get_multi_synonyms("./configs/cls_potsdam.txt")
print(f"Single-word queries: {len(SINGLE_WORDS)}", flush=True)
print(f"Multi-synonym queries: {len(MULTI_WORDS)} synonyms -> 5 classes", flush=True)

# ── cache text embeddings ──────────────────────────────────────────────────
print("Caching text embeddings...", flush=True)

def cache_text(words, with_soft=False):
    cache = []
    with torch.no_grad():
        for word in words:
            te = model.backbone.forward_text([word], device=DEVICE)
            if with_soft and soft_prompts is not None:
                lang_f = te["language_features"]
                soft_m = torch.zeros(1, soft_prompts.shape[0], device=DEVICE,
                                     dtype=te["language_mask"].dtype)
                te["language_features"] = torch.cat([lang_f, soft_prompts.to(dtype=lang_f.dtype)], dim=0)
                te["language_mask"]     = torch.cat([te["language_mask"], soft_m], dim=1)
            cache.append({k: v.cpu() for k, v in te.items()})
    return cache

cache_single = cache_text(SINGLE_WORDS,  with_soft=False)
cache_multi  = cache_text(MULTI_WORDS,   with_soft=False)
cache_ptsam  = cache_text(MULTI_WORDS,   with_soft=True) if soft_prompts is not None else None
print("Text caching done.", flush=True)

# ── label helpers ──────────────────────────────────────────────────────────
RGB_TO_IDX = {
    (255,255,255): 0, (0,0,255): 1, (0,255,255): 2,
    (0,255,0): 3,     (255,255,0): 4, (255,0,0): 5,
}
def rgb_to_gt(path):
    arr = tifffile.imread(str(path))
    if arr.ndim == 2:
        gt = torch.from_numpy(arr.astype(np.int64))
        if gt.max() > 5: gt = gt - 1
        gt[gt < 0] = IGNORE_IDX
        return gt
    h, w = arr.shape[:2]
    gt = torch.full((h, w), IGNORE_IDX, dtype=torch.int64)
    for (r, g, b), idx in RGB_TO_IDX.items():
        m = (arr[:,:,0]==r) & (arr[:,:,1]==g) & (arr[:,:,2]==b)
        gt[m] = idx
    return gt

# ── load tile ──────────────────────────────────────────────────────────────
img_path = next(POTSDAM_ROOT.rglob(f"*{VAL_TILE}*_RGB.tif"), None)
lbl_path = next(POTSDAM_ROOT.rglob(f"*{VAL_TILE}*_label_noBoundary.tif"), None)
if img_path is None: raise SystemExit("ERROR: val tile not found")

img_arr = tifffile.imread(str(img_path))[:,:,:3]
gt_full  = rgb_to_gt(lbl_path)
H_full, W_full = img_arr.shape[:2]
print(f"Tile {VAL_TILE}: {W_full}x{H_full}  valid GT: {(gt_full!=IGNORE_IDX).sum():,}", flush=True)

# ── score assembly ─────────────────────────────────────────────────────────
def collect_class_scores(state, h, w, te_cache, cls_idx_list):
    logits = torch.zeros((N_CLASSES, h, w), device=DEVICE)
    for te_cpu, cls_idx in zip(te_cache, cls_idx_list):
        processor.reset_all_prompts(state)
        for k, v in te_cpu.items(): state["backbone_out"][k] = v.to(DEVICE)
        state["geometric_prompt"] = model._get_dummy_prompt()
        processor._forward_grounding(state)
        scores = torch.zeros((h, w), device=DEVICE)
        if state.get("masks_logits") is not None and state["masks_logits"].shape[0] > 0:
            for i in range(state["masks_logits"].shape[0]):
                il = state["masks_logits"][i].squeeze()
                if il.shape != (h, w):
                    il = F.interpolate(il.view(1,1,*il.shape), size=(h,w),
                                       mode="bilinear", align_corners=False).squeeze()
                scores = torch.max(scores, il * state["object_score"][i])
        sem = state["semantic_mask_logits"].squeeze()
        if sem.shape != (h, w):
            sem = F.interpolate(sem.view(1,1,*sem.shape), size=(h,w),
                                mode="bilinear", align_corners=False).squeeze()
        scores = torch.max(scores, sem) * state["presence_score"]
        logits[cls_idx] = torch.max(logits[cls_idx], scores)
    return logits

# ── Gaussian kernel ────────────────────────────────────────────────────────
def make_gaussian_kernel(h, w, dev):
    sy, sx = h/4.0, w/4.0
    y = torch.arange(h, device=dev).float() - (h-1)/2.0
    x = torch.arange(w, device=dev).float() - (w-1)/2.0
    return torch.exp(-y[:,None]**2/(2*sy**2)) * torch.exp(-x[None,:]**2/(2*sx**2))

# ── sliding window ─────────────────────────────────────────────────────────
h_grids = max(H_full - CROP_SIZE + STRIDE - 1, 0) // STRIDE + 1
w_grids = max(W_full - CROP_SIZE + STRIDE - 1, 0) // STRIDE + 1
total   = h_grids * w_grids
print(f"Sliding window: {h_grids}x{w_grids}={total} crops", flush=True)

gauss_k    = make_gaussian_kernel(CROP_SIZE, CROP_SIZE, DEVICE)
# accumulators for 3 raw prediction sets (single, multi, ptsam)
acc_single = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE)
acc_multi  = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE)
acc_ptsam  = torch.zeros(N_CLASSES, H_full, W_full, device=DEVICE) if cache_ptsam else None
wt_mat     = torch.zeros(H_full, W_full, device=DEVICE)

for hi in range(h_grids):
    for wi in range(w_grids):
        y1 = hi*STRIDE;  x1 = wi*STRIDE
        y2 = min(y1+CROP_SIZE, H_full);  x2 = min(x1+CROP_SIZE, W_full)
        y1 = max(y2-CROP_SIZE, 0);       x1 = max(x2-CROP_SIZE, 0)

        crop_pil = Image.fromarray(img_arr[y1:y2, x1:x2])
        h_c, w_c = y2-y1, x2-x1

        with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
            state = processor.set_image(crop_pil)
            l_single = collect_class_scores(state, h_c, w_c, cache_single, SINGLE_CLS_IDX).float()
            l_multi  = collect_class_scores(state, h_c, w_c, cache_multi,  MULTI_CLS_IDX).float()
            if cache_ptsam:
                l_ptsam = collect_class_scores(state, h_c, w_c, cache_ptsam, MULTI_CLS_IDX).float()

        g = gauss_k[:h_c, :w_c]
        acc_single[:, y1:y2, x1:x2] += l_single * g.unsqueeze(0)
        acc_multi[:,  y1:y2, x1:x2] += l_multi  * g.unsqueeze(0)
        if cache_ptsam:
            acc_ptsam[:, y1:y2, x1:x2] += l_ptsam * g.unsqueeze(0)
        wt_mat[y1:y2, x1:x2] += g

        done = hi*w_grids + wi + 1
        if done % 10 == 0 or done == total:
            print(f"  {done}/{total} crops", flush=True)

# ── normalize and finalize ─────────────────────────────────────────────────
def normalize(acc):
    return acc / wt_mat.unsqueeze(0)  # [N_CLS, H, W], soft logits

soft_single = normalize(acc_single)
soft_multi  = normalize(acc_multi)
soft_ptsam  = normalize(acc_ptsam) if acc_ptsam is not None else None

# ── PAMR refinement ────────────────────────────────────────────────────────
def apply_pamr(soft_logits):
    """Refine soft segmentation logits using RGB image as affinity guide."""
    img_t = torch.from_numpy(img_arr.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    logits_in = soft_logits.unsqueeze(0)  # [1, N_CLS, H, W]
    with torch.no_grad():
        refined = pamr_module(img_t, logits_in)
    return refined.squeeze(0)

print("Running PAMR on multi and ptsam...", flush=True)
soft_multi_pamr = apply_pamr(soft_multi)
soft_ptsam_pamr = apply_pamr(soft_ptsam) if soft_ptsam is not None else None
print("PAMR done.", flush=True)

def finalize(soft):
    seg = soft.argmax(0)
    seg[soft.max(0)[0] < PROB_THD] = BG_IDX
    return seg.cpu()

seg_A = finalize(soft_single)           # ZS-single
seg_B = finalize(soft_multi)            # ZS-multi
seg_C = finalize(soft_multi_pamr)       # ZS-multi + PAMR
seg_D = finalize(soft_ptsam_pamr) if soft_ptsam_pamr is not None else None  # PTSAM + PAMR

# ── mIoU (5-class, excl. clutter) ─────────────────────────────────────────
EVAL_CLASSES = [c for c in range(N_CLASSES) if c != BG_IDX]
CLS_NAMES    = ["impervious", "building", "low_veg", "tree", "car", "clutter"]

def compute_miou(seg_pred, gt):
    tp = torch.zeros(N_CLASSES); fp = torch.zeros(N_CLASSES); fn = torch.zeros(N_CLASSES)
    valid = (gt != IGNORE_IDX)
    for c in range(N_CLASSES):
        pc = (seg_pred == c) & valid
        lc = (gt       == c) & valid
        tp[c] = (pc & lc).sum().float()
        fp[c] = (pc & ~lc).sum().float()
        fn[c] = (~pc & lc).sum().float()
    iou  = tp / (tp + fp + fn + 1e-6)
    macc = (tp / (tp + fn + 1e-6))[EVAL_CLASSES].mean().item() * 100
    return iou, iou[EVAL_CLASSES].mean().item() * 100, macc

METHODS = {
    "A  ZS-single":    seg_A,
    "B  ZS-multi":     seg_B,
    "C  ZS-multi+PAMR": seg_C,
    "D  PTSAM+PAMR":   seg_D,
}

results = {}
print(f"\n=== Tile {VAL_TILE} — ablation (5-class mIoU, excl. clutter) ===")
print(f"  {'method':<20} ", end="")
for n in CLS_NAMES[:5]: print(f"{n:>11}", end="")
print(f"  {'mIoU':>8}  {'mAcc':>8}  {'vs_lit':>8}")
print(f"  {'-'*95}")
for name, seg in METHODS.items():
    if seg is None:
        print(f"  {name:<20}  (skipped — soft_prompts not found)")
        continue
    iou, miou, macc = compute_miou(seg, gt_full)
    results[name] = (iou, miou, macc)
    row = f"  {name:<20} "
    for c in EVAL_CLASSES: row += f"{iou[c]*100:10.2f}%"
    row += f"  {miou:7.2f}%  {macc:7.2f}%  {miou-57.8:+7.2f}%"
    print(row)
print(f"  {'-'*95}")
print(f"  Literature (SegEarth-OV-3 zero-shot): 57.8%")

# ── visualization ──────────────────────────────────────────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PALETTE = np.array([[255,255,255],[0,0,255],[0,255,255],[0,255,0],[255,255,0],[255,0,0]], dtype=np.uint8)

def to_rgb(label_np):
    out = np.zeros((*label_np.shape, 3), dtype=np.uint8)
    for c in range(N_CLASSES): out[label_np == c] = PALETTE[c]
    out[label_np == IGNORE_IDX] = [80, 80, 80]
    return out

from PIL import Image as PILImage
SCALE = max(H_full, W_full) / 1500
def thumb(arr):
    h, w = arr.shape[:2]
    return np.array(PILImage.fromarray(arr).resize((int(w/SCALE), int(h/SCALE)), PILImage.NEAREST))

gt_rgb = to_rgb(gt_full.numpy())
segs   = [seg_A, seg_B, seg_C, seg_D]
seg_rgbs = [to_rgb(s.numpy()) if s is not None else gt_rgb for s in segs]

def get_miou(name):
    return f"{results[name][1]:.1f}%" if name in results else "n/a"

col_titles = [
    "RGB",
    "Ground Truth",
    f"A  ZS-single\n{get_miou('A  ZS-single')}",
    f"B  ZS-multi\n{get_miou('B  ZS-multi')}",
    f"C  ZS+PAMR\n{get_miou('C  ZS-multi+PAMR')}",
    f"D  PTSAM+PAMR\n{get_miou('D  PTSAM+PAMR')}",
]

CROPS = [
    ("top-left",     0,             0),
    ("centre",       H_full//2-512, W_full//2-512),
    ("bottom-right", H_full-1024,   W_full-1024),
]

fig, axes = plt.subplots(4, 6, figsize=(36, 24),
                         gridspec_kw={"hspace": 0.05, "wspace": 0.02})
for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=11, fontweight="bold", pad=8)

imgs_row0 = [thumb(img_arr), thumb(gt_rgb)] + [thumb(s) for s in seg_rgbs]
for j, im in enumerate(imgs_row0):
    axes[0, j].imshow(im); axes[0, j].axis("off")
axes[0, 0].set_ylabel("full tile", fontsize=11)

for row, (lbl, y1c, x1c) in enumerate(CROPS, start=1):
    y1c = max(0, min(y1c, H_full - 1024))
    x1c = max(0, min(x1c, W_full - 1024))
    y2c, x2c = y1c + 1024, x1c + 1024
    crops = [img_arr[y1c:y2c, x1c:x2c], gt_rgb[y1c:y2c, x1c:x2c]] + \
            [s[y1c:y2c, x1c:x2c] for s in seg_rgbs]
    for j, im in enumerate(crops):
        axes[row, j].imshow(im); axes[row, j].axis("off")
    axes[row, 0].set_ylabel(lbl, fontsize=11)

legend_patches = [mpatches.Patch(color=PALETTE[i]/255., label=CLS_NAMES[i]) for i in range(N_CLASSES)]
fig.legend(handles=legend_patches, loc="lower center", ncol=N_CLASSES,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, 0.01))

miou_summary = "  |  ".join(
    f"{n.split()[0]}: {results[n][1]:.1f}%" for n in METHODS if n in results
)
fig.suptitle(
    f"Tile {VAL_TILE} Ablation (5-class, lit=57.8%)\n{miou_summary}",
    fontsize=13, fontweight="bold", y=1.002)

out_png = Path("/kaggle/working/ablation_comparison.png")
fig.savefig(str(out_png), dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved -> {out_png}", flush=True)
PYEOF